## Category Init

**Purpose:** Loads the shared PySpark utilities used throughout this notebook — column expression helpers (`F`), type definitions for schema casting, and common column functions (`trim`, `col`, `when`, `coalesce`, `lit`) used in the transformation and insight sections below.

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql.functions import trim, col, when, coalesce, lit

## Reading from the Bronze Table

**Purpose:** Loads the raw category-level sales data from the Bronze layer (`workspace.bronze.categories_data`) into a Spark DataFrame for cleaning.

**Schema:** `Category` (string), `Items_sold` (int), `Net_Revenue` (double), `Products` (int), `Orders` (int) — one row per product category with its lifetime sales totals.

In [0]:
df = spark.table('workspace.bronze.categories_data')

## Data Transformations

**Purpose:** Cleans the bronze data and promotes it to the Silver layer.

**Steps:**
1. Trim leading/trailing whitespace from `Category` names.
2. Overwrite `workspace.silver.silver_categories_data` with the cleaned DataFrame.

**Note:** The silver table still holds 238 rows with only 232 distinct category names (duplicate names and near-duplicates such as the DTF adhesive powder variants), so the insight cells below consolidate by `Category` before ranking.

In [0]:
# df.display()

df_clean = df.withColumn('Category', trim(col('Category')))

df_clean.write.mode('overwrite').saveAsTable('workspace.silver.silver_categories_data')

df_clean.display()


## Category Insights

**Purpose:** Turns the cleaned silver data into business answers. Data is first consolidated by category (one row per `Category`, summing items, revenue, products, and orders), then analysed from four angles:

1. **Overall KPIs** — portfolio-wide totals and averages
2. **Top 10 categories by revenue** — the biggest revenue drivers
3. **Pareto (80/20) analysis** — how concentrated revenue is across categories
4. **Efficiency metrics** — revenue per item, per order, and basket size

### 1. Overall KPIs

One-row summary of the full category portfolio: total categories, items sold, net revenue, orders, and products, plus the average revenue per category. Duplicate category names are merged by grouping on `Category` first.

In [0]:
from pyspark.sql.window import Window

silver_df = spark.table('workspace.silver.silver_categories_data')

# Aggregate by Category first: 238 rows hold only 232 distinct category names,
# so duplicate names must be consolidated before ranking.
cat_df = (silver_df
    .groupBy('Category')
    .agg(
        F.sum('Items_sold').alias('items_sold'),
        F.sum('Net_Revenue').alias('net_revenue'),
        F.sum('Products').alias('products'),
        F.sum('Orders').alias('orders'),
    )
)

kpi_df = cat_df.agg(
    F.count('*').alias('total_categories'),
    F.sum('items_sold').alias('total_items_sold'),
    F.round(F.sum('net_revenue'), 2).alias('total_net_revenue'),
    F.sum('orders').alias('total_orders'),
    F.sum('products').alias('total_products'),
    F.round(F.avg('net_revenue'), 2).alias('avg_revenue_per_category'),
)

display(kpi_df)

### 2. Top 10 Categories by Revenue

The ten highest-grossing categories with their share of total net revenue. Heat Transfer Vinyl alone drives ~19% of all revenue.

In [0]:
# Top 10 categories by net revenue
top_revenue_df = (cat_df
    .select('Category', 'net_revenue', 'items_sold', 'orders', 'products')
    .orderBy(F.col('net_revenue').desc())
    .limit(10)
    .withColumn('pct_of_total_revenue', F.round(F.col('net_revenue') / F.sum('net_revenue').over(Window.orderBy(F.lit(1))) * 100, 2))
)

display(top_revenue_df)

### 3. Pareto Analysis (80/20)

Ranks categories by revenue and accumulates their contribution to answer: *how many categories generate 80% of total net revenue?* Result: 40 of 232 categories (~17%) produce 80% of revenue — revenue is heavily concentrated.

In [0]:
# Pareto analysis: how many categories generate 80% of total revenue
pareto_df = (cat_df
    .withColumn('revenue_rank', F.row_number().over(Window.orderBy(F.col('net_revenue').desc())))
    .withColumn('cum_revenue', F.sum('net_revenue').over(Window.orderBy('revenue_rank').rowsBetween(Window.unboundedPreceding, Window.currentRow)))
    .withColumn('cum_revenue_pct', F.round(F.col('cum_revenue') / F.sum('net_revenue').over(Window.orderBy(F.lit(1))) * 100, 2))
)

pareto_80 = pareto_df.filter(F.col('cum_revenue_pct') <= 80)
pareto_80.display()

print(f"{pareto_80.count()} of {cat_df.count()} categories generate 80% of total net revenue")

### 4. Efficiency Metrics

Revenue per item sold, revenue per order, and items per order. This view surfaces high-ticket equipment categories (exposure units, heat presses, plotters at $1K–$2.7K per item) versus bulk consumables that rely on volume.

In [0]:
# Efficiency metrics: revenue per item sold and revenue per order,
# highlighting the most monetarily efficient categories
revenue_per_item = (cat_df
    .withColumn('revenue_per_item', F.round(F.col('net_revenue') / F.col('items_sold'), 2))
    .withColumn('revenue_per_order', F.round(F.col('net_revenue') / F.col('orders'), 2))
    .withColumn('items_per_order', F.round(F.col('items_sold') / F.col('orders'), 2))
    .orderBy(F.col('revenue_per_item').desc())
)

display(revenue_per_item.limit(10))